In [10]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_curve, auc, precision_recall_curve, average_precision_score
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import seaborn as sns
from deap import base, creator, tools, algorithms
import random

# Load NSL-KDD Dataset
def load_nsl_kdd():
    column_names = [
        'duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes', 'land', 'wrong_fragment',
        'urgent', 'hot', 'num_failed_logins', 'logged_in', 'num_compromised', 'root_shell', 'su_attempted',
        'num_root', 'num_file_creations', 'num_shells', 'num_access_files', 'num_outbound_cmds', 'is_host_login',
        'is_guest_login', 'count', 'srv_count', 'serror_rate', 'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate',
        'same_srv_rate', 'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count', 'dst_host_same_srv_rate',
        'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate', 'dst_host_srv_diff_host_rate', 'dst_host_serror_rate',
        'dst_host_srv_serror_rate', 'dst_host_rerror_rate', 'dst_host_srv_rerror_rate', 'class', 'drop'
    ]
    
    # Load the dataset
    train_data = pd.read_csv('KDDTrain+.txt', names=column_names)
    test_data = pd.read_csv('KDDTest+.txt', names=column_names)
    train_data.drop('drop', axis=1, inplace=True)
    test_data.drop('drop', axis=1, inplace=True)


    return train_data, test_data

def preprocess_data(train_data, test_data):
    # Encoding categorical features
    label_encoder = LabelEncoder()
    categorical_columns = ['protocol_type', 'service', 'flag']
    for col in categorical_columns:
        train_data[col] = label_encoder.fit_transform(train_data[col])
        test_data[col] = label_encoder.transform(test_data[col])

    # Map attack types to 0 (Normal) and 1 (Attack)
    train_data['class'] = train_data['class'].apply(lambda x: 1 if x != 'normal.' else 0)
    test_data['class'] = test_data['class'].apply(lambda x: 1 if x != 'normal.' else 0)

    # Check the class distribution
    print("Training set class distribution:")
    print(train_data['class'].value_counts())

    print("Test set class distribution:")
    print(test_data['class'].value_counts())

    # Check for empty classes in either training or test set
    if len(train_data['class'].unique()) == 1:
        print("Warning: Training set has only one class.")
    if len(test_data['class'].unique()) == 1:
        print("Warning: Test set has only one class.")
        
    # Extract features and target variable
    X_train = train_data.drop(columns='class')
    y_train = train_data['class']
    X_test = test_data.drop(columns='class')
    y_test = test_data['class']

    return X_train, y_train, X_test, y_test

from imblearn.over_sampling import SMOTE

# Apply SMOTE to balance the training data
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

# Now, you can train the model on the resampled dataset
print("Resampled training set class distribution:")
print(pd.Series(y_train_resampled).value_counts())


print("Training set class distribution:")
print(y_train.value_counts())

print("Test set class distribution:")
print(y_test.value_counts())


# Genetic Algorithm for Hyperparameter Tuning
def create_individual():
    # Random values for XGBoost hyperparameters
    max_depth = random.randint(3, 10)
    learning_rate = random.uniform(0.01, 0.2)
    n_estimators = random.randint(50, 200)
    colsample_bytree = random.uniform(0.5, 1)
    subsample = random.uniform(0.5, 1)

    return [max_depth, learning_rate, n_estimators, colsample_bytree, subsample]

def evaluate_individual(individual):
    max_depth, learning_rate, n_estimators, colsample_bytree, subsample = individual

    # Initialize and train XGBoost model
    model = xgb.XGBClassifier(
        max_depth=int(max_depth),
        learning_rate=learning_rate,
        n_estimators=n_estimators,
        colsample_bytree=colsample_bytree,
        subsample=subsample
    )
    model.fit(X_train, y_train)

    # Predictions and accuracy score
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)

    return accuracy,

# Set up Genetic Algorithm
creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", list, fitness=creator.FitnessMax)

toolbox = base.Toolbox()
toolbox.register("individual", tools.initIterate, creator.Individual, create_individual)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)
toolbox.register("mate", tools.cxBlend, alpha=0.5)
toolbox.register("mutate", tools.mutGaussian, mu=0.0, sigma=1.0, indpb=0.2)
toolbox.register("select", tools.selTournament, tournsize=3)
toolbox.register("evaluate", evaluate_individual)

# Genetic Algorithm Main Function
def run_genetic_algorithm():
    population = toolbox.population(n=10)
    algorithms.eaSimple(population, toolbox, cxpb=0.7, mutpb=0.2, ngen=5, verbose=True)
    return population

# Model Evaluation and Metrics Plotting
def plot_metrics(model, X_test, y_test):
    # Confusion Matrix
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Normal', 'Attack'], yticklabels=['Normal', 'Attack'])
    plt.title("Confusion Matrix")
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.show()

    # ROC Curve
    y_prob = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)
    plt.figure()
    plt.plot(fpr, tpr, color='blue', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
    plt.plot([0, 1], [0, 1], color='gray', lw=2, linestyle='--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Receiver Operating Characteristic (ROC) Curve')
    plt.legend(loc='lower right')
    plt.show()

    # Precision-Recall Curve
    precision, recall, _ = precision_recall_curve(y_test, y_prob)
    avg_precision = average_precision_score(y_test, y_prob)
    plt.figure()
    plt.plot(recall, precision, color='green', lw=2, label=f'Precision-Recall curve (average precision = {avg_precision:.2f})')
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title('Precision-Recall Curve')
    plt.legend(loc='lower left')
    plt.show()

# Main Execution
train_data, test_data = load_nsl_kdd()
X_train, y_train, X_test, y_test = preprocess_data(train_data, test_data)

# Run genetic algorithm to optimize XGBoost hyperparameters
population = run_genetic_algorithm()

# Select the best individual
best_individual = tools.selBest(population, 1)[0]
print(f'Best Hyperparameters: {best_individual}')

# Train the XGBoost model using the best hyperparameters
best_model = xgb.XGBClassifier(
    max_depth=int(best_individual[0]),
    learning_rate=best_individual[1],
    n_estimators=best_individual[2],
    colsample_bytree=best_individual[3],
    subsample=best_individual[4]
)
best_model.fit(X_train, y_train)

# Model evaluation and plotting metrics
plot_metrics(best_model, X_test, y_test)


ValueError: The target 'y' needs to have more than 1 class. Got 1 class instead